In [ ]:
import mlflow.pyfunc

class CatboostModelProba(mlflow.pyfunc.PythonModel):

    def __init__(self, model):
        super().__init__()
        self._model = model

    def predict(self, context, model_input):
        probas = self._model.predict(model_input) ** 0.5

        return probas

from sklearn.tree import DecisionTreeClassifier
custom_model = CatboostModelProba(model)

In [ ]:
experiment_id = "0"
custom_model = CustomMlflowModel(
    model
)  # ваш код инициализации модели через CustomMlflowModel


with mlflow.start_run(
    run_name="custom_model",
    experiment_id=experiment_id,
) as run:

    model_info = mlflow.pyfunc.log_model(
        python_model=custom_model, artifact_path="models_custom_model"
    )  # напишите код логирования модели тут

Регистрации модели вместе с окружением

In [ ]:
import mlflow
import numpy as np

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

# напишите код, который подключает tracking и registry uri
mlflow.set_tracking_uri(
    f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
)  # tracking uri
mlflow.set_registry_uri(
    f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
)  # registry uri

experiment_id = "0"

# указываем путь до окружения
pip_requirements = "../requirements.txt"

# формируем сигнатуру, дополнительно передавая параметры применения модели
signature = mlflow.models.infer_signature(
    np.array([[0.1, 0.2, 0.3], [0.1, 0.2, 0.3]]), np.array([0.1, 0.2])
)
# формируем пример входных данных
input_example = [[0.1, 0.2, 0.3], [0.1, 0.2, 0.3]]
# предположим, мы хотим указать на то, что модель предсказывает на месяц вперёд
metadata = {"target_name": "churn"}
# путь до скрипта или ноутбука, который осуществляет обучение модели и валидацию
code_paths = ["train.py", "val_model.py"]


with mlflow.start_run(
    run_name="model_reg", experiment_id=experiment_id
) as run:
    run_id = run.info.run_id

    model_info = mlflow.catboost.log_model(
        # ваш код здесь #
        cb_model=model,
        artifact_path="models",
        registered_model_name=REGISTRY_MODEL_NAME,
        pip_requirements=pip_requirements,
        signature=signature,
        input_example=input_example,
        metadata=metadata,
        code_path=code_paths,
    )